In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.export import export


In [ ]:
class SimpleCNNF(nn.Module):
    def __init__(self):
        super(SimpleCNNF, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))  # Output: (16, 14, 14)
        x = F.relu(self.conv2(x))  # Output: (32, 7, 7)
        return x


In [ ]:
from utils.drawer import draw_graph
# Create model and sample input
model = SimpleCNNF()
model.eval()

# Export to FX graph
sample_input = torch.randn(1, 1, 28, 28)
exported_program = export(model, (sample_input,), strict=False)
graph_module = exported_program.graph_module

draw_graph(graph_module)


In [ ]:
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from mxpt2e.quantizer import MXQuantizer, get_symmetric_quantization_mx_config
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    get_symmetric_quantization_config,
    XNNPACKQuantizer,
)

quantizer = MXQuantizer()
quantizer.set_global(
    get_symmetric_quantization_mx_config(observer_type="minmax", block_size=32, quantize_weights=True, is_per_channel=True)
    # get_symmetric_quantization_config()
)
prepared = prepare_pt2e(graph_module, quantizer)


In [ ]:
draw_graph(prepared)


In [ ]:
converted = convert_pt2e(prepared, use_reference_representation=False)


In [ ]:
draw_graph(converted)


In [ ]:
from mxpt2e.convert_pass import insert_mx_quant_dequant_after_int24_dq
converted_mx = insert_mx_quant_dequant_after_int24_dq(converted, block_size=32)


In [ ]:
draw_graph(converted_mx)

In [ ]:
torch.ops.quantized_decomposed.dequantize_per_channel

In [ ]:
print(dir(torch.ops.quantized_decomposed.dequantize_per_channel))

In [ ]:
torch.ops.quantized_decomposed.dequantize_per_channel.Tensor